In [ ]:
%pip install pyarrow

In [1]:
from pathlib import Path

import re
import pandas as pd
import numpy as np

In [2]:
def normalize_gateway_id(x):
    """
    Convert gateway IDs into one canonical 12-hex-character format.

    Examples:
        06:39:EA:56:02:C1 -> 0639EA5602C1
        06-39-EA-56-02-C1 -> 0639EA5602C1
        0639EA5602C1      -> 0639EA5602C1
    """

    if pd.isna(x):
        return np.nan

    # Convert to string and normalize case/whitespace
    x = str(x).strip().upper()

    # Keep only hexadecimal characters
    x = re.sub(r"[^0-9A-F]", "", x)

    return x

In [3]:
# ============================================================
# CONFIGURATION
# ============================================================

# Current notebook folder
NOTEBOOK_DIR = Path.cwd()

# Project root
PROJECT_DIR = NOTEBOOK_DIR.parent

# Data folder
DATA_DIR = PROJECT_DIR / "Data"

TELEMETRY_DIR = DATA_DIR / "telemetry"
MASTER_FILE = DATA_DIR / "gateway_master.csv"
METER_FILE = DATA_DIR / "meter_read_success.csv"
VISITS_FILE = DATA_DIR / "field_visits.csv"

HISTORICAL_OUTPUT = DATA_DIR / "historical_gateway_week_dataset.csv"
CHALLENGE_OUTPUT = DATA_DIR / "challenge_gateway_week_dataset.csv"


# ------------------------------------------------------------
# Feature history
# ------------------------------------------------------------
# We need 28 days of history before making a prediction.

LOOKBACK_DAYS = 28


# ------------------------------------------------------------
# Challenge prediction period
# ------------------------------------------------------------

CHALLENGE_START = pd.Timestamp("2026-02-02")
CHALLENGE_END = pd.Timestamp("2026-03-23")


CHALLENGE_WEEKS = pd.date_range(
    start=CHALLENGE_START,
    end=CHALLENGE_END,
    freq="7D"
)


# ------------------------------------------------------------
# Historical prediction weeks
# ------------------------------------------------------------
# We start after enough telemetry history exists.
# Telemetry starts around 2025-08-01.
# Therefore September is the first useful period.

HISTORICAL_START = pd.Timestamp("2025-09-01")

HISTORICAL_END = pd.Timestamp("2026-01-26")

HISTORICAL_WEEKS = pd.date_range(
    start=HISTORICAL_START,
    end=HISTORICAL_END,
    freq="7D"
)


print("Historical weeks:")
print(HISTORICAL_WEEKS)

print("\nChallenge weeks:")
print(CHALLENGE_WEEKS)

Historical weeks:
DatetimeIndex(['2025-09-01', '2025-09-08', '2025-09-15', '2025-09-22',
               '2025-09-29', '2025-10-06', '2025-10-13', '2025-10-20',
               '2025-10-27', '2025-11-03', '2025-11-10', '2025-11-17',
               '2025-11-24', '2025-12-01', '2025-12-08', '2025-12-15',
               '2025-12-22', '2025-12-29', '2026-01-05', '2026-01-12',
               '2026-01-19', '2026-01-26'],
              dtype='datetime64[ns]', freq='7D')

Challenge weeks:
DatetimeIndex(['2026-02-02', '2026-02-09', '2026-02-16', '2026-02-23',
               '2026-03-02', '2026-03-09', '2026-03-16', '2026-03-23'],
              dtype='datetime64[ns]', freq='7D')


In [4]:
print("Data folder exists:", DATA_DIR.exists())
print("Telemetry folder exists:", TELEMETRY_DIR.exists())
print("Gateway master exists:", MASTER_FILE.exists())
print("Meter file exists:", METER_FILE.exists())
print("Field visits exists:", VISITS_FILE.exists())

Data folder exists: True
Telemetry folder exists: True
Gateway master exists: True
Meter file exists: True
Field visits exists: True


In [5]:
# ============================================================
# LOAD TELEMETRY
# ============================================================

telemetry_columns = [
    "gateway_id",
    "ts_utc",

    # Traffic
    "rx_nr_pkts",
    "rx_crc_bad",
    "tx_success",
    "tx_busy",
    "tx_override",
    "number_of_messages",

    # CPU / memory
    "avg_idletime",
    "avg_load1",
    "load1_bigger1",
    "load1_bigger2",
    "avg_memfree",
    "avg_uptime",
    "avg_activeproccess",
    "avg_totalproccess",

    # Reboots
    "reboot_cnt",
    "reboot_duration_sec",
    "r_cnt_power_cycle",
    "r_cnt_reboot",
    "r_cnt_unknown",
    "r_dur_power_cycle",
    "r_dur_reboot",
    "r_dur_unknown",
    "avg_reboot_duration",
    "reboot_importance",

    # Connectivity
    "disconnection_cnt",
    "offline_duration_sec",
    "avg_offline_duration",
    "online_duration_mins",
    "no_conn_importance",

    # Network
    "network_2g",
    "network_3g",
    "network_4g",
    "network_unknown",

    # Operators
    "operator_unknown",
    "operator_3AT",
    "operator_A1",
    "operator_Eplus",
    "operator_O2DE",
    "operator_OrangeLU",
    "operator_Salt",
    "operator_Swisscom",
    "operator_TmobileA",
    "operator_TelekomDE",
    "operator_VodafoneDE",

    # Signal
    "rssi_good",
    "rssi_normal",
    "rssi_bad",

    "rscp_rsrp_good",
    "rscp_rsrp_normal",
    "rscp_rsrp_bad",

    "ecio_rsrq_good",
    "ecio_rsrq_normal",
    "ecio_rsrq_bad",
]


telemetry = pd.read_parquet(
    TELEMETRY_DIR,
    columns=telemetry_columns
)

telemetry["gateway_id"] = (
    telemetry["gateway_id"]
    .map(normalize_gateway_id)
)

# Convert timestamp
telemetry["ts"] = pd.to_datetime(
    telemetry["ts_utc"],
    utc=True
)


# Remove original timestamp
telemetry.drop(
    columns=["ts_utc"],
    inplace=True
)


# Sort
telemetry.sort_values(
    ["gateway_id", "ts"],
    inplace=True
)


print("Telemetry shape:", telemetry.shape)

print(
    "Telemetry date range:",
    telemetry["ts"].min(),
    "→",
    telemetry["ts"].max()
)

print(
    "Number of gateways:",
    telemetry["gateway_id"].nunique()
)

Telemetry shape: (1433387, 55)
Telemetry date range: 2025-08-01 00:00:00+00:00 → 2026-03-31 23:00:00+00:00
Number of gateways: 320


In [6]:
# ============================================================
# LOAD GATEWAY MASTER
# ============================================================

master = pd.read_csv(
    MASTER_FILE,
    encoding="latin1"
)


master_columns = [
    "gateway_id",
    "tenant",
    "site_type",
    "region",
    "hw_model",
    "antenna_type",
    "fw_version",
    "n_meters_installed",
]

master["gateway_id"] = (
    master["gateway_id"]
    .map(normalize_gateway_id)
)

master = master[
    [c for c in master_columns if c in master.columns]
]

master = master.drop_duplicates(
    subset=["gateway_id"]
).reset_index(drop=True)


print("Master shape:", master.shape)

master.head()

print(
    "Master unique gateways:",
    master["gateway_id"].nunique()
)

Master shape: (332, 8)
Master unique gateways: 332


In [7]:
# ============================================================
# LOAD METER READ SUCCESS
# ============================================================

meters = pd.read_csv(
    METER_FILE,
    encoding="latin1"
)

meters["gateway_id"] = (
    meters["gateway_id"]
    .map(normalize_gateway_id)
)


meters["week_start"] = pd.to_datetime(
    meters["week_start"]
)


meters["meter_success_rate"] = (
    meters["meters_read"]
    /
    meters["meters_expected"].replace(0, np.nan)
)


print("Meter shape:", meters.shape)

print(
    "Meter date range:",
    meters["week_start"].min(),
    "→",
    meters["week_start"].max()
)


meters.head()

Meter shape: (7226, 5)
Meter date range: 2025-08-04 00:00:00 → 2026-01-26 00:00:00


,week_start,gateway_id,meters_expected,meters_read,meter_success_rate
0,2025-08-04,0202CB0A6B1F,166,156,0.939759
1,2025-08-04,02043B6BA08B,475,433,0.911579
2,2025-08-04,02075B45BF24,146,121,0.828767
3,2025-08-04,02091DF7CCB8,128,95,0.742188
4,2025-08-04,02097AB58D3C,315,276,0.876190


In [8]:
# ============================================================
# LOAD FIELD VISITS
# ============================================================

visits = pd.read_csv(
    VISITS_FILE,
    encoding="latin1"
)

visits["gateway_id"] = (
    visits["gateway_id"]
    .map(normalize_gateway_id)
)

visits["requested_on"] = pd.to_datetime(
    visits["requested_on"],
    errors="coerce"
)


visits["visited_on"] = pd.to_datetime(
    visits["visited_on"],
    errors="coerce"
)


print("Visits shape:", visits.shape)

print(
    "Visit date range:",
    visits["requested_on"].min(),
    "→",
    visits["requested_on"].max()
)


visits.head()

Visits shape: (642, 8)
Visit date range: 2025-02-03 00:00:00 → 2026-01-30 00:00:00


,visit_id,gateway_id,requested_on,visited_on,reason_reported,outcome,parts_replaced,technician_hours
0,WO-2025-00001,0230EEF72435,2025-02-03,2025-02-05,Auffaellige Statistik,Kein Fehler gefunden,NaN,1.28
1,WO-2025-00007,02CFFA3A3F35,2025-02-03,2025-02-12,Signal schwach,Kein Fehler gefunden,NaN,0.64
2,WO-2025-00004,06C0F27E810B,2025-02-03,2025-02-14,Keine Verbindung,Kein Fehler gefunden,NaN,1.65
3,WO-2025-00003,0A0B47664A85,2025-02-04,2025-02-11,Haeufige Neustarts,Kein Fehler gefunden,NaN,1.08
4,WO-2025-00008,0EE09EDA649F,2025-02-04,2025-02-17,Auffaellige Statistik,Kein Zugang,NaN,0.93


In [9]:
# ============================================================
# VERIFY GATEWAY ID MAPPING
# ============================================================

telemetry_ids = set(
    telemetry["gateway_id"].dropna()
)

master_ids = set(
    master["gateway_id"].dropna()
)

print("Telemetry unique IDs :", len(telemetry_ids))
print("Master unique IDs    :", len(master_ids))
print("Common IDs           :", len(telemetry_ids & master_ids))
print("Telemetry only       :", len(telemetry_ids - master_ids))
print("Master only          :", len(master_ids - telemetry_ids))

Telemetry unique IDs : 320
Master unique IDs    : 332
Common IDs           : 320
Telemetry only       : 0
Master only          : 12


In [10]:
# ============================================================
# VERIFY ACTUAL PANDAS MERGE
# ============================================================

test_week = pd.Timestamp("2025-09-01")

end = test_week.tz_localize("UTC")
start = end - pd.Timedelta(days=7)

test_window = telemetry[
    (telemetry["ts"] >= start) &
    (telemetry["ts"] < end)
].copy()

test_features = (
    test_window
    .groupby("gateway_id")
    .agg(
        offline_mean=("offline_duration_sec", "mean"),
        reboot_sum=("reboot_cnt", "sum"),
        disconnect_sum=("disconnection_cnt", "sum"),
        load_mean=("avg_load1", "mean")
    )
    .reset_index()
)

print("Telemetry feature rows:", len(test_features))
print(
    "Telemetry feature IDs:",
    test_features["gateway_id"].nunique()
)

test_merge = master[["gateway_id"]].drop_duplicates().merge(
    test_features,
    on="gateway_id",
    how="left"
)

print("\nMaster gateways:", len(test_merge))
print(
    "Matched gateways:",
    test_merge["offline_mean"].notna().sum()
)
print(
    "Unmatched gateways:",
    test_merge["offline_mean"].isna().sum()
)

Telemetry feature rows: 280
Telemetry feature IDs: 280

Master gateways: 332
Matched gateways: 280
Unmatched gateways: 52


In [13]:
# ============================================================
# TELEMETRY FEATURE ENGINEERING
# ============================================================

def aggregate_telemetry(telemetry, prediction_week):

    telemetry = telemetry.copy()

    # --------------------------------------------------------
    # Normalize IDs
    # --------------------------------------------------------

    telemetry["gateway_id"] = (
        telemetry["gateway_id"]
        .map(normalize_gateway_id)
    )

    # --------------------------------------------------------
    # Prediction week → UTC
    # --------------------------------------------------------

    prediction_week = pd.Timestamp(prediction_week)

    if prediction_week.tzinfo is None:
        end = prediction_week.tz_localize("UTC")
    else:
        end = prediction_week.tz_convert("UTC")

    # --------------------------------------------------------
    # Define windows
    # --------------------------------------------------------

    windows = {
        "7d": end - pd.Timedelta(days=7),
        "14d": end - pd.Timedelta(days=14),
        "28d": end - pd.Timedelta(days=28),
    }

    result = None

    # ========================================================
    # PROCESS WINDOWS
    # ========================================================

    for window_name, start in windows.items():

        data = telemetry[
            (telemetry["ts"] >= start) &
            (telemetry["ts"] < end)
        ].copy()

        print(
            f"{window_name}: "
            f"{len(data):,} rows, "
            f"{data['gateway_id'].nunique()} gateways"
        )

        if data.empty:
            continue

        # ----------------------------------------------------
        # Aggregate by gateway
        # ----------------------------------------------------

        features = (
            data
            .groupby("gateway_id")
            .agg(

                telemetry_hours=("ts", "count"),

                # ----------------------------
                # Connectivity
                # ----------------------------

                offline_duration_sec_sum=(
                    "offline_duration_sec", "sum"
                ),
                offline_duration_sec_mean=(
                    "offline_duration_sec", "mean"
                ),
                offline_duration_sec_max=(
                    "offline_duration_sec", "max"
                ),

                disconnection_cnt_sum=(
                    "disconnection_cnt", "sum"
                ),
                disconnection_cnt_mean=(
                    "disconnection_cnt", "mean"
                ),
                disconnection_cnt_max=(
                    "disconnection_cnt", "max"
                ),

                avg_offline_duration_mean=(
                    "avg_offline_duration", "mean"
                ),
                online_duration_mins_mean=(
                    "online_duration_mins", "mean"
                ),

                no_conn_importance_mean=(
                    "no_conn_importance", "mean"
                ),
                no_conn_importance_max=(
                    "no_conn_importance", "max"
                ),

                # ----------------------------
                # Reboots
                # ----------------------------

                reboot_cnt_sum=(
                    "reboot_cnt", "sum"
                ),
                reboot_cnt_mean=(
                    "reboot_cnt", "mean"
                ),
                reboot_cnt_max=(
                    "reboot_cnt", "max"
                ),

                reboot_duration_sec_sum=(
                    "reboot_duration_sec", "sum"
                ),

                reboot_importance_mean=(
                    "reboot_importance", "mean"
                ),
                reboot_importance_max=(
                    "reboot_importance", "max"
                ),

                r_cnt_power_cycle_sum=(
                    "r_cnt_power_cycle", "sum"
                ),
                r_cnt_reboot_sum=(
                    "r_cnt_reboot", "sum"
                ),
                r_cnt_unknown_sum=(
                    "r_cnt_unknown", "sum"
                ),

                # ----------------------------
                # CPU / Memory
                # ----------------------------

                avg_load1_mean=(
                    "avg_load1", "mean"
                ),
                avg_load1_max=(
                    "avg_load1", "max"
                ),

                load1_bigger1_sum=(
                    "load1_bigger1", "sum"
                ),
                load1_bigger2_sum=(
                    "load1_bigger2", "sum"
                ),

                avg_memfree_mean=(
                    "avg_memfree", "mean"
                ),
                avg_memfree_min=(
                    "avg_memfree", "min"
                ),

                avg_uptime_mean=(
                    "avg_uptime", "mean"
                ),

                # ----------------------------
                # Traffic
                # ----------------------------

                rx_nr_pkts_sum=(
                    "rx_nr_pkts", "sum"
                ),
                rx_crc_bad_sum=(
                    "rx_crc_bad", "sum"
                ),

                tx_success_sum=(
                    "tx_success", "sum"
                ),
                tx_busy_sum=(
                    "tx_busy", "sum"
                ),
                tx_override_sum=(
                    "tx_override", "sum"
                ),

                number_of_messages_sum=(
                    "number_of_messages", "sum"
                ),

                # ----------------------------
                # RSSI
                # ----------------------------

                rssi_good_sum=(
                    "rssi_good", "sum"
                ),
                rssi_normal_sum=(
                    "rssi_normal", "sum"
                ),
                rssi_bad_sum=(
                    "rssi_bad", "sum"
                ),

                # ----------------------------
                # RSRP / RSCP
                # ----------------------------

                rscp_rsrp_good_sum=(
                    "rscp_rsrp_good", "sum"
                ),
                rscp_rsrp_normal_sum=(
                    "rscp_rsrp_normal", "sum"
                ),
                rscp_rsrp_bad_sum=(
                    "rscp_rsrp_bad", "sum"
                ),

                # ----------------------------
                # ECIO / RSRQ
                # ----------------------------

                ecio_rsrq_good_sum=(
                    "ecio_rsrq_good", "sum"
                ),
                ecio_rsrq_normal_sum=(
                    "ecio_rsrq_normal", "sum"
                ),
                ecio_rsrq_bad_sum=(
                    "ecio_rsrq_bad", "sum"
                ),

                # ----------------------------
                # Network
                # ----------------------------

                network_2g_sum=(
                    "network_2g", "sum"
                ),
                network_3g_sum=(
                    "network_3g", "sum"
                ),
                network_4g_sum=(
                    "network_4g", "sum"
                ),
                network_unknown_sum=(
                    "network_unknown", "sum"
                ),
            )
            .reset_index()
        )

        # ----------------------------------------------------
        # Normalize IDs after aggregation
        # ----------------------------------------------------

        features["gateway_id"] = (
            features["gateway_id"]
            .map(normalize_gateway_id)
        )

        # ----------------------------------------------------
        # Add suffix
        # ----------------------------------------------------

        features.rename(
            columns={
                c: f"{c}_{window_name}"
                for c in features.columns
                if c != "gateway_id"
            },
            inplace=True
        )

        # ----------------------------------------------------
        # Merge windows
        # ----------------------------------------------------

        if result is None:

            result = features

        else:

            result = result.merge(
                features,
                on="gateway_id",
                how="outer"
            )

    # ========================================================
    # NO DATA
    # ========================================================

    if result is None:
        return pd.DataFrame()

    # ========================================================
    # SIGNAL RATIOS
    # ========================================================

    def safe_ratio(numerator, denominator):
        return numerator / denominator.replace(0, np.nan)

    # RSSI

    rssi_total = (
        result["rssi_good_sum_7d"]
        + result["rssi_normal_sum_7d"]
        + result["rssi_bad_sum_7d"]
    )

    result["rssi_bad_ratio_7d"] = safe_ratio(
        result["rssi_bad_sum_7d"],
        rssi_total
    )

    # RSRP

    rsrp_total = (
        result["rscp_rsrp_good_sum_7d"]
        + result["rscp_rsrp_normal_sum_7d"]
        + result["rscp_rsrp_bad_sum_7d"]
    )

    result["rscp_rsrp_bad_ratio_7d"] = safe_ratio(
        result["rscp_rsrp_bad_sum_7d"],
        rsrp_total
    )

    # ECIO

    ecio_total = (
        result["ecio_rsrq_good_sum_7d"]
        + result["ecio_rsrq_normal_sum_7d"]
        + result["ecio_rsrq_bad_sum_7d"]
    )

    result["ecio_rsrq_bad_ratio_7d"] = safe_ratio(
        result["ecio_rsrq_bad_sum_7d"],
        ecio_total
    )

    # ========================================================
    # TRENDS
    # ========================================================

    def relative_change(recent, historical):
        return (
            (recent - historical)
            / (historical.abs() + 1e-6)
        )

    result["offline_trend"] = relative_change(
        result["offline_duration_sec_mean_7d"],
        result["offline_duration_sec_mean_28d"]
    )

    result["reboot_trend"] = relative_change(
        result["reboot_cnt_mean_7d"],
        result["reboot_cnt_mean_28d"]
    )

    result["disconnect_trend"] = relative_change(
        result["disconnection_cnt_mean_7d"],
        result["disconnection_cnt_mean_28d"]
    )

    result["load_trend"] = relative_change(
        result["avg_load1_mean_7d"],
        result["avg_load1_mean_28d"]
    )

    result["memory_trend"] = relative_change(
        result["avg_memfree_mean_7d"],
        result["avg_memfree_mean_28d"]
    )

    return result

In [14]:
# ============================================================
# VISIT FEATURES + TARGET
# ============================================================

def create_visit_features(
    gateway_ids,
    prediction_week,
    visits
):

    week = pd.Timestamp(prediction_week)

    rows = []

    for gateway_id in gateway_ids:

        gateway_visits = visits[
            visits["gateway_id"] == gateway_id
        ]

        # ----------------------------------------------------
        # PAST VISITS ONLY
        # ----------------------------------------------------

        past_visits = gateway_visits[
            gateway_visits["requested_on"] < week
        ]

        last_30 = past_visits[
            past_visits["requested_on"]
            >= week - pd.Timedelta(days=30)
        ]

        last_90 = past_visits[
            past_visits["requested_on"]
            >= week - pd.Timedelta(days=90)
        ]

        visits_last_30d = len(last_30)
        visits_last_90d = len(last_90)

        has_previous_visit = int(len(past_visits) > 0)

        if len(past_visits) > 0:

            last_visit = past_visits[
                "requested_on"
            ].max()

            days_since_last_visit = (
                week - last_visit
            ).days

        else:

            days_since_last_visit = np.nan

        # ----------------------------------------------------
        # FUTURE TARGET
        # ----------------------------------------------------

        next_week = gateway_visits[
            (gateway_visits["requested_on"] >= week)
            &
            (
                gateway_visits["requested_on"]
                < week + pd.Timedelta(days=7)
            )
        ]

        visit_next_7d = int(
            len(next_week) > 0
        )

        rows.append({
            "gateway_id": gateway_id,
            "visits_last_30d": visits_last_30d,
            "visits_last_90d": visits_last_90d,
            "days_since_last_visit": days_since_last_visit,
            "has_previous_visit": has_previous_visit,
            "visit_next_7d": visit_next_7d
        })

    return pd.DataFrame(rows)

In [15]:
# ============================================================
# CREATE ONE GATEWAY-WEEK DATASET
# ============================================================

def create_week_dataset(
    telemetry,
    master,
    meters,
    visits,
    prediction_week
):

    week = pd.Timestamp(prediction_week)

    print(f"\nCreating week: {week.date()}")

    # --------------------------------------------------------
    # Start with gateway universe
    # --------------------------------------------------------

    week_df = master.copy()

    week_df["gateway_id"] = (
        week_df["gateway_id"]
        .map(normalize_gateway_id)
    )

    week_df["week_start"] = week

    # --------------------------------------------------------
    # Telemetry
    # --------------------------------------------------------

    telemetry_features = aggregate_telemetry(
        telemetry,
        week
    )

    if not telemetry_features.empty:

        telemetry_features["gateway_id"] = (
            telemetry_features["gateway_id"]
            .map(normalize_gateway_id)
        )

        week_df = week_df.merge(
            telemetry_features,
            on="gateway_id",
            how="left",
            validate="one_to_one"
        )
        
        week_df["telemetry_missing_7d"] = (
            week_df["offline_duration_sec_mean_7d"]
            .isna()
            .astype(int)
        )

        week_df["telemetry_missing_14d"] = (
            week_df["offline_duration_sec_mean_14d"]
            .isna()
            .astype(int)
        )

        week_df["telemetry_missing_28d"] = (
            week_df["offline_duration_sec_mean_28d"]
            .isna()
            .astype(int)
        )


        week_df["telemetry_coverage_7d"] = (
            week_df["telemetry_hours_7d"] / 168
        ).clip(0, 1)

        week_df["telemetry_coverage_14d"] = (
            week_df["telemetry_hours_14d"] / 336
        ).clip(0, 1)

        week_df["telemetry_coverage_28d"] = (
            week_df["telemetry_hours_28d"] / 672
        ).clip(0, 1)

    # --------------------------------------------------------
    # Previous completed meter week
    # --------------------------------------------------------

    previous_meters = meters[
        meters["week_start"] < week
    ].copy()

    if not previous_meters.empty:

        previous_meters = (
            previous_meters
            .sort_values("week_start")
            .groupby("gateway_id")
            .tail(1)
        )

        meter_columns = [
            "gateway_id",
            "meters_expected",
            "meters_read",
            "meter_success_rate"
        ]

        week_df = week_df.merge(
            previous_meters[meter_columns],
            on="gateway_id",
            how="left",
            validate="one_to_one"
        )

    # --------------------------------------------------------
    # Historical visit features + target
    # --------------------------------------------------------

    visit_features = create_visit_features(
        week_df["gateway_id"].tolist(),
        week,
        visits
    )

    week_df = week_df.merge(
        visit_features,
        on="gateway_id",
        how="left",
        validate="one_to_one"
    )

    return week_df

In [16]:
# ============================================================
# GENERATE HISTORICAL DATASET
# ============================================================

historical_data = []

for week in HISTORICAL_WEEKS:

    df_week = create_week_dataset(
        telemetry,
        master,
        meters,
        visits,
        week
    )
    
    print(
        "Telemetry coverage:",
        df_week["offline_duration_sec_mean_7d"].notna().sum(),
        "/",
        len(df_week)
    )

    historical_data.append(df_week)


historical_df = pd.concat(
    historical_data,
    ignore_index=True
)


historical_df.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)


historical_df.sort_values(
    ["week_start", "gateway_id"],
    inplace=True
)


historical_df.reset_index(
    drop=True,
    inplace=True
)


print("Historical dataset shape:")
print(historical_df.shape)


Creating week: 2025-09-01
7d: 40,744 rows, 280 gateways
14d: 81,703 rows, 280 gateways
28d: 163,852 rows, 280 gateways
Telemetry coverage: 280 / 332

Creating week: 2025-09-08
7d: 40,956 rows, 280 gateways
14d: 81,700 rows, 280 gateways
28d: 163,747 rows, 280 gateways
Telemetry coverage: 280 / 332

Creating week: 2025-09-15
7d: 43,266 rows, 280 gateways
14d: 84,222 rows, 280 gateways
28d: 165,925 rows, 280 gateways
Telemetry coverage: 280 / 332

Creating week: 2025-09-22
7d: 40,850 rows, 280 gateways
14d: 84,116 rows, 280 gateways
28d: 165,816 rows, 280 gateways
Telemetry coverage: 280 / 332

Creating week: 2025-09-29
7d: 40,693 rows, 279 gateways
14d: 81,543 rows, 280 gateways
28d: 165,765 rows, 280 gateways
Telemetry coverage: 279 / 332

Creating week: 2025-10-06
7d: 40,575 rows, 279 gateways
14d: 81,268 rows, 279 gateways
28d: 165,384 rows, 280 gateways
Telemetry coverage: 279 / 332

Creating week: 2025-10-13
7d: 40,351 rows, 278 gateways
14d: 80,926 rows, 279 gateways
28d: 162,469

In [17]:
# ============================================================
# GENERATE CHALLENGE DATASET
# ============================================================

challenge_data = []

for week in CHALLENGE_WEEKS:

    df_week = create_week_dataset(
        telemetry,
        master,
        meters,
        visits,
        week
    )

    challenge_data.append(df_week)


challenge_df = pd.concat(
    challenge_data,
    ignore_index=True
)


challenge_df.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)


challenge_df.sort_values(
    ["week_start", "gateway_id"],
    inplace=True
)


challenge_df.reset_index(
    drop=True,
    inplace=True
)


print("Challenge dataset shape:")
print(challenge_df.shape)


Creating week: 2026-02-02
7d: 41,653 rows, 290 gateways
14d: 85,085 rows, 290 gateways
28d: 164,913 rows, 290 gateways

Creating week: 2026-02-09
7d: 42,088 rows, 292 gateways
14d: 83,741 rows, 292 gateways
28d: 167,291 rows, 292 gateways

Creating week: 2026-02-16
7d: 42,354 rows, 294 gateways
14d: 84,442 rows, 296 gateways
28d: 169,527 rows, 296 gateways

Creating week: 2026-02-23
7d: 42,707 rows, 298 gateways
14d: 85,061 rows, 298 gateways
28d: 168,802 rows, 300 gateways

Creating week: 2026-03-02
7d: 43,136 rows, 300 gateways
14d: 85,843 rows, 300 gateways
28d: 170,285 rows, 302 gateways

Creating week: 2026-03-09
7d: 43,429 rows, 303 gateways
14d: 86,565 rows, 304 gateways
28d: 171,626 rows, 304 gateways

Creating week: 2026-03-16
7d: 44,301 rows, 308 gateways
14d: 87,730 rows, 308 gateways
28d: 173,573 rows, 309 gateways

Creating week: 2026-03-23
7d: 44,627 rows, 308 gateways
14d: 88,928 rows, 308 gateways
28d: 175,493 rows, 309 gateways
Challenge dataset shape:
(2656, 169)


In [18]:
# ============================================================
# SAVE
# ============================================================

historical_df.to_csv(
    HISTORICAL_OUTPUT,
    index=False
)

challenge_df.to_csv(
    CHALLENGE_OUTPUT,
    index=False
)


print("Saved:")
print(HISTORICAL_OUTPUT)
print(CHALLENGE_OUTPUT)

Saved:
/Users/udaykumarreddy/Documents/Gateway-Ranking/Data/historical_gateway_week_dataset.csv
/Users/udaykumarreddy/Documents/Gateway-Ranking/Data/challenge_gateway_week_dataset.csv


In [19]:
# ============================================================
# VERIFY HISTORICAL DATA
# ============================================================

print("Shape:", historical_df.shape)

print("\nWeeks:")
print(
    historical_df["week_start"]
    .drop_duplicates()
    .sort_values()
    .to_list()
)

print("\nRows per week:")
display(
    historical_df
    .groupby("week_start")
    .size()
    .to_frame("rows")
)

print("\nTarget distribution:")
display(
    historical_df["visit_next_7d"]
    .value_counts()
    .to_frame("count")
)

print("\nFirst rows:")
display(
    historical_df.head()
)

Shape: (7304, 169)

Weeks:
[Timestamp('2025-09-01 00:00:00'), Timestamp('2025-09-08 00:00:00'), Timestamp('2025-09-15 00:00:00'), Timestamp('2025-09-22 00:00:00'), Timestamp('2025-09-29 00:00:00'), Timestamp('2025-10-06 00:00:00'), Timestamp('2025-10-13 00:00:00'), Timestamp('2025-10-20 00:00:00'), Timestamp('2025-10-27 00:00:00'), Timestamp('2025-11-03 00:00:00'), Timestamp('2025-11-10 00:00:00'), Timestamp('2025-11-17 00:00:00'), Timestamp('2025-11-24 00:00:00'), Timestamp('2025-12-01 00:00:00'), Timestamp('2025-12-08 00:00:00'), Timestamp('2025-12-15 00:00:00'), Timestamp('2025-12-22 00:00:00'), Timestamp('2025-12-29 00:00:00'), Timestamp('2026-01-05 00:00:00'), Timestamp('2026-01-12 00:00:00'), Timestamp('2026-01-19 00:00:00'), Timestamp('2026-01-26 00:00:00')]

Rows per week:


,rows
week_start,
2025-09-01,332
2025-09-08,332
2025-09-15,332
2025-09-22,332
2025-09-29,332
2025-10-06,332
2025-10-13,332
2025-10-20,332
2025-10-27,332



Target distribution:


,count
visit_next_7d,
0,7039
1,265



First rows:


,gateway_id,tenant,site_type,region,hw_model,antenna_type,fw_version,n_meters_installed,week_start,telemetry_hours_7d,...,telemetry_coverage_14d,telemetry_coverage_28d,meters_expected,meters_read,meter_success_rate,visits_last_30d,visits_last_90d,days_since_last_visit,has_previous_visit,visit_next_7d
0,0202CB0A6B1F,tenant_a,Gebäude,Sachsen,GW-2100,Omni 3dBi,3.2.0,166,2025-09-01,158.0,...,0.940476,0.947917,166.0,153.0,0.921687,0,1,31.0,1,0
1,02043B6BA08B,tenant_b,Gebäude,Hessen,GW-2100L,Panel 7dBi,3.2.0,475,2025-09-01,151.0,...,0.922619,0.885417,475.0,463.0,0.974737,0,0,145.0,1,0
2,02075B45BF24,tenant_a,Gebäude,Nordrhein-Westfalen,GW-2100,Omni 3dBi,2.15.1,146,2025-09-01,153.0,...,0.886905,0.891369,146.0,124.0,0.849315,0,1,56.0,1,0
3,02091DF7CCB8,tenant_a,Gebäude,Baden-Württemberg,GW-2100,Yagi 9dBi,2.15.1,128,2025-09-01,120.0,...,0.705357,0.699405,128.0,98.0,0.765625,0,1,53.0,1,1
4,02097AB58D3C,tenant_a,Gebäude,Baden-Württemberg,GW-2100,Omni 3dBi,3.2.0,315,2025-09-01,158.0,...,0.940476,0.919643,315.0,296.0,0.939683,0,1,31.0,1,0


In [20]:
# ============================================================
# VERIFY CHALLENGE DATA
# ============================================================

print("Shape:", challenge_df.shape)

print("\nRows per challenge week:")

display(
    challenge_df
    .groupby("week_start")
    .size()
    .to_frame("rows")
)

print("\nChallenge weeks:")

display(
    challenge_df["week_start"]
    .drop_duplicates()
    .sort_values()
)

Shape: (2656, 169)

Rows per challenge week:


,rows
week_start,
2026-02-02,332
2026-02-09,332
2026-02-16,332
2026-02-23,332
2026-03-02,332
2026-03-09,332
2026-03-16,332
2026-03-23,332



Challenge weeks:


0      2026-02-02
332    2026-02-09
664    2026-02-16
996    2026-02-23
1328   2026-03-02
1660   2026-03-09
1992   2026-03-16
2324   2026-03-23
Name: week_start, dtype: datetime64[ns]

In [21]:
print("Telemetry minimum:")
print(telemetry["ts"].min())

print("\nTelemetry maximum:")
print(telemetry["ts"].max())

Telemetry minimum:
2025-08-01 00:00:00+00:00

Telemetry maximum:
2026-03-31 23:00:00+00:00


In [22]:
week = pd.Timestamp("2025-09-01", tz="UTC")

start = week - pd.Timedelta(days=7)

test = telemetry[
    (telemetry["ts"] >= start) &
    (telemetry["ts"] < week)
]

print("Lookback:", start, "to", week)
print("Rows found:", len(test))
print("Gateways found:", test["gateway_id"].nunique())

print("\nTelemetry gateway IDs:")
print(telemetry["gateway_id"].head(10).tolist())

print("\nTest gateway IDs:")
print(test["gateway_id"].head(10).tolist())

print("\nSample telemetry:")
print(test.head())

Lookback: 2025-08-25 00:00:00+00:00 to 2025-09-01 00:00:00+00:00
Rows found: 40744
Gateways found: 280

Telemetry gateway IDs:
['0202CB0A6B1F', '0202CB0A6B1F', '0202CB0A6B1F', '0202CB0A6B1F', '0202CB0A6B1F', '0202CB0A6B1F', '0202CB0A6B1F', '0202CB0A6B1F', '0202CB0A6B1F', '0202CB0A6B1F']

Test gateway IDs:
['0202CB0A6B1F', '0202CB0A6B1F', '0202CB0A6B1F', '0202CB0A6B1F', '0202CB0A6B1F', '0202CB0A6B1F', '0202CB0A6B1F', '0202CB0A6B1F', '0202CB0A6B1F', '0202CB0A6B1F']

Sample telemetry:
          gateway_id  rx_nr_pkts  rx_crc_bad  tx_success  tx_busy  \
132335  0202CB0A6B1F          42          40           0        0   
132336  0202CB0A6B1F          33          32           0        0   
132337  0202CB0A6B1F          29          29           0        0   
132338  0202CB0A6B1F          33          33           0        0   
132339  0202CB0A6B1F          31          29           0        0   

        tx_override  number_of_messages  avg_idletime  avg_load1  \
132335            0           

In [23]:
test_week = create_week_dataset(
    telemetry,
    master,
    meters,
    visits,
    pd.Timestamp("2025-09-01")
)

print("Shape:", test_week.shape)

print(
    "Telemetry populated:",
    test_week["offline_duration_sec_mean_7d"].notna().sum()
)

print(
    "Telemetry missing:",
    test_week["offline_duration_sec_mean_7d"].isna().sum()
)


Creating week: 2025-09-01
7d: 40,744 rows, 280 gateways
14d: 81,703 rows, 280 gateways
28d: 163,852 rows, 280 gateways
Shape: (332, 169)
Telemetry populated: 280
Telemetry missing: 52


In [24]:
df = pd.read_csv(HISTORICAL_OUTPUT)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nDuplicate gateway-week rows:")
print(
    df.duplicated(subset=["gateway_id", "week_start"]).sum()
)

Shape: (7304, 169)

Columns:
['gateway_id', 'tenant', 'site_type', 'region', 'hw_model', 'antenna_type', 'fw_version', 'n_meters_installed', 'week_start', 'telemetry_hours_7d', 'offline_duration_sec_sum_7d', 'offline_duration_sec_mean_7d', 'offline_duration_sec_max_7d', 'disconnection_cnt_sum_7d', 'disconnection_cnt_mean_7d', 'disconnection_cnt_max_7d', 'avg_offline_duration_mean_7d', 'online_duration_mins_mean_7d', 'no_conn_importance_mean_7d', 'no_conn_importance_max_7d', 'reboot_cnt_sum_7d', 'reboot_cnt_mean_7d', 'reboot_cnt_max_7d', 'reboot_duration_sec_sum_7d', 'reboot_importance_mean_7d', 'reboot_importance_max_7d', 'r_cnt_power_cycle_sum_7d', 'r_cnt_reboot_sum_7d', 'r_cnt_unknown_sum_7d', 'avg_load1_mean_7d', 'avg_load1_max_7d', 'load1_bigger1_sum_7d', 'load1_bigger2_sum_7d', 'avg_memfree_mean_7d', 'avg_memfree_min_7d', 'avg_uptime_mean_7d', 'rx_nr_pkts_sum_7d', 'rx_crc_bad_sum_7d', 'tx_success_sum_7d', 'tx_busy_sum_7d', 'tx_override_sum_7d', 'number_of_messages_sum_7d', 'rssi_g

In [ ]:
print("Total columns:", len(test_week.columns))

display(
    pd.DataFrame({
        "column": test_week.columns,
        "dtype": test_week.dtypes.astype(str)
    })
)

In [ ]:
print("\nMissing percentage:")
display(
    (
        test_week.isna().mean() * 100
    )
    .sort_values(ascending=False)
    .head(30)
    .to_frame("missing_percent")
)

In [ ]:
# ============================================================
# TRAIN / VALIDATION / TEST SPLIT
# ============================================================

historical_df["week_start"] = pd.to_datetime(
    historical_df["week_start"]
)


TRAIN_END = pd.Timestamp("2025-12-22")

VALIDATION_END = pd.Timestamp("2026-01-12")

TEST_END = pd.Timestamp("2026-01-26")


train_df = historical_df[
    historical_df["week_start"] <= TRAIN_END
].copy()


validation_df = historical_df[
    (historical_df["week_start"] > TRAIN_END)
    &
    (historical_df["week_start"] <= VALIDATION_END)
].copy()


test_df = historical_df[
    (historical_df["week_start"] > VALIDATION_END)
    &
    (historical_df["week_start"] <= TEST_END)
].copy()


print("TRAIN:", train_df.shape)

print("VALIDATION:", validation_df.shape)

print("TEST:", test_df.shape)